# 基于指针的C语言编程算子开发教程

## 概述

课程延续中级教程中的基于指针的C语言编程算子开发教程，以Softmax算子为例，继续讲解基于指针的C语言编程的算子优化方法。通过本课程可学习到如下知识点：
- Double Buffer的原理及对性能的影响。
- 数据搬运性能优化的常见方法。

## 课程结构

1. 环境准备
2. 前置课程Softmax算子性能分析
3. 高阶版Softmax算子实现
4. 优化前后的性能对比
5. 课后实践


---
## 1. 环境准备

正式开始学习之前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及使用bisheng编译器，完成算子的开发及编译。

In [ ]:
import os
import subprocess

!mkdir -p src

result = subprocess.run(
    ['bash', '-l', '-c', 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'],
    capture_output=True, text=True
)
for line in result.stdout.strip().split('\n'):
    line = line.strip()
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value
print("\n🎉 Environment initialization process completed successfully!")

---
## 2. 前置课程Softmax算子性能分析

**图1** 前置课程进阶版Softmax算子流水图

<img src="./images/07.03_softmax/softmax_improved_timeline.png" title="进阶版流水图" style="zoom:100%;" />

进一步查看详细性能数据：
|   | vector核总时间 | PIPE_V时间 | PIPE_V时间占比 | PIPE_S时间 | PIPE_S时间占比 | PIPE_MTE2时间 | PIPE_MTE2时间占比 | PIPE_MTE3时间 | PIPE_MTE3时间占比 |
| :-----: | :-------: | :-------: | :-------: | :-------: | :-------: | :-------: | :-------: | :-------: | :-------: |
| 进阶版本 | 4261.62  | 643.671 | 15.1% | 552.156 | 13% | 2938.025 | 68.9% | 1080.408 | 25.4% |

从上述流水图可以得出以下规律：
- 算子的每一轮VF计算均按照搬入 -> 计算 -> 搬出的顺序执行。
- 通过src_ub与dst_ub资源分别管理的方式实现了前一轮VF计算的PIPE_MTE2和后一轮VF计算的PIPE_MTE3的并行。
- 由于src_ub与dst_ub的唯一性，PIPE_V无法和PIPE_MTE2/PIPE_MTE3并行。
- PIPE_MTE2/PIPE_MTE3耗时明显多于PIPE_V耗时。

由此，我们可以初步设定优化方向：
- 进一步提高并行度，让PIPE_V和PIPE_MTE2/PIPE_MTE3能够并行执行。
- 提高数据搬运操作的效率，缩短计算类操作和搬运类操作耗时的时间差。减少整体耗时。

### 2.1 使能Double Buffer机制

Double Buffer是基于MTE指令队列与Vector指令队列的独立性和可并行性，通过将数据搬运与Vector计算并行执行以隐藏大部分的数据搬运时间，并降低Vector指令的等待时间，最终提高Vector单元的利用效率。

使能后的流水示意如下：

**图2** 使能Double Buffer的流水示意图

<img src="./images/07.03_softmax/double_buff_timeline.png" title="使能Double Buffer的流水示意图" style="zoom:100%;" />

注意，使用Double Buffer需占用更多的UB空间。因此，需要减少单个VF处理的数据量。该操作会降低VF内部双发能力带来的优化效果。实际应用中需结合实际情况，合理划分数据。

针对本例，应优先考虑降低搬运开销。将单VF处理数据行数降至40行，循环次数增加至2048次。保持单核总数据量不变。

```c
constexpr uint32_t LOOP_COUNT = 2048;
constexpr uint32_t WIDTH = 128;
constexpr uint32_t SINGLE_VF_HEIGHT = 40;
constexpr uint32_t SINGLE_VF_DATA_LEN = SINGLE_VF_HEIGHT * WIDTH;
```

在C API中，我们需要定义两份UB内存。搬运、计算操作均在在两份内存上交替执行。

```c
    // 申请双份UB内存
    __ubuf__ float src0_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float dst0_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float exp0_ub[SINGLE_VF_DATA_LEN];

    __ubuf__ float src1_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float dst1_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float exp1_ub[SINGLE_VF_DATA_LEN];

    uint8_t mutex_id_src0 = 1;   // src0_ub对应的资源锁
    uint8_t mutex_id_dst0 = 2;   // dst0_ub对应的资源锁
    uint8_t mutex_id_src1 = 3;   // src1_ub对应的资源锁
    uint8_t mutex_id_dst1 = 4;   // dst1_ub对应的资源锁

    for (uint32_t i = 0; i < LOOP_COUNT; i++) {
        // 当次使用的UB资源
        __ubuf__ float* src_ub;
        __ubuf__ float* dst_ub;
        __ubuf__ float* exp_ub;
        uint8_t mutex_id_src;   // src_ub对应的资源锁
        uint8_t mutex_id_dst;   // dst_ub对应的资源锁

        if(i % 2 == 0) {
            src_ub = src0_ub;
            dst_ub = dst0_ub;
            exp_ub = exp0_ub;
            mutex_id_src = mutex_id_src0;
            mutex_id_dst = mutex_id_dst0;
        } else {
            src_ub = src1_ub;
            dst_ub = dst1_ub;
            exp_ub = exp1_ub;
            mutex_id_src = mutex_id_src1;
            mutex_id_dst = mutex_id_dst1;
        }

        // 计算操作
    }
```

### 2.2 提高搬运指令效率

搬运不同大小的数据块时，对带宽的利用率不一样。根据实测经验，单次搬运数据长度16KB以上时，通常能较好地发挥出带宽的最佳性能。因此对于单次搬运，应考虑尽可能的搬运较大的数据块。

前置课程样例中，通过for循环逐行搬运。增加了指令数，和scalar操作，令MTE流水的执行效率降低。一般可以通过调整asc_copy_gm2ub_align/asc_copy_ub2gm_align API中的数据块数量、数据块长度、步长等参数实现连续/非连续的数据搬运。

另外，softmax算子的输入/输出数据均为一次性读写操作，不存在重复读写的场景。可通过l2_cache_mode参数关闭l2缓存，读写操作均直接在UB/GM上进行，避免不必要的cache读写开销，提高读写效率。

注意，l2_cache_mode参数需结合算子特点选择，不可盲目关闭。

```c
// 本例为连续搬入，第三入参数据块数设置为1，第4入参数据块长度设置为单个VF的数据长度。由于只有一个数据块，最后两个步长参数设置为0即可。
// 倒数第3入参设置为4（NOTALLOC_KEEP），表示不启用L2 Cache，每次都直接从GM中读取，并且保持已有Cache Line的状态不变。
asc_copy_gm2ub_align(src_ub, &x_gm[i * SINGLE_VF_DATA_LEN], 1, SINGLE_VF_DATA_LEN * sizeof(float), 0, 0, true, 4, 0, 0);

// 本例为连续搬出，第三入参数据块数设置为1，第4入参数据块长度设置为单个VF的数据长度。由于只有一个数据块，最后两个步长参数设置为0即可。
// 倒数第3入参设置为4（NOTALLOC_CLEAN），表示不启用L2 Cache，若L2 Cache中已有同地址缓存会被保留并标记为Clean，标记为Clean的Cache Line被移出L2 Cache时会被直接丢弃。
asc_copy_ub2gm_align(&y_gm[i * SINGLE_VF_DATA_LEN], dst_ub, 1, SINGLE_VF_DATA_LEN * sizeof(float), 4, 0, 0);
```

## 3. 高阶版Softmax算子实现

### 3.1 softmax.h文件实现如下：

In [ ]:
%%writefile src/softmax.h

#include "c_api/asc_simd.h"

namespace Custom {
union DataUnion {
    constexpr __aicore__ DataUnion() : f(0.0f) {}
    constexpr __aicore__ DataUnion(uint32_t val) : i(val) {}
    float f;
    uint32_t i;
};
constexpr DataUnion fp32_min_value(0x00800000u);
}

constexpr uint32_t LOOP_COUNT = 2048;
constexpr uint32_t WIDTH = 128;
constexpr uint32_t SINGLE_VF_HEIGHT = 40;
constexpr uint32_t SINGLE_VF_DATA_LEN = SINGLE_VF_HEIGHT * WIDTH;
constexpr uint32_t SINGLE_CORE_DATA_LEN = SINGLE_VF_DATA_LEN * LOOP_COUNT;

__simd_vf__ inline void softmax_vf(__ubuf__ float* dst_ub, __ubuf__ float* src_ub, __ubuf__ float* exp_ub)
{
    constexpr uint16_t one_repeat_cnt = asc_get_vf_len() / sizeof(float);
    uint16_t repeat_times = (WIDTH + one_repeat_cnt - 1) / one_repeat_cnt;
    vector_bool mask_full = asc_create_mask_b32(PAT_ALL);

    vector_float src_reg;
    vector_float max_reg;
    vector_float exp_reg;
    vector_float sum_reg;
    vector_float div_reg;

    vector_float src_reg1;
    vector_float max_reg1;
    vector_float exp_reg1;
    vector_float sum_reg1;
    vector_float div_reg1;

    uint16_t halfA = SINGLE_VF_HEIGHT >> 1;

    // 第一部分: ReduceMax -> Sub -> Exp
    for (uint16_t i = 0; i < halfA; i++) {
        asc_duplicate_scalar(max_reg, Custom::fp32_min_value.f, mask_full);
        asc_duplicate_scalar(max_reg1, Custom::fp32_min_value.f, mask_full);
        for (uint16_t j = 0; j < repeat_times; j++) {
            asc_loadalign(src_reg, src_ub + i * WIDTH + j * one_repeat_cnt);
            asc_loadalign(src_reg1, src_ub + i * WIDTH + j * one_repeat_cnt + halfA * WIDTH);
            asc_max(max_reg, max_reg, src_reg, mask_full);
            asc_max(max_reg1, max_reg1, src_reg1, mask_full);
        }
        asc_reduce_max(max_reg, max_reg, mask_full);
        asc_reduce_max(max_reg1, max_reg1, mask_full);
        asc_duplicate(max_reg, max_reg, mask_full);
        asc_duplicate(max_reg1, max_reg1, mask_full);
        for (uint16_t j = 0; j < repeat_times; j++) {
            asc_loadalign(src_reg, src_ub + i * WIDTH + j * one_repeat_cnt);
            asc_loadalign(src_reg1, src_ub + i * WIDTH + j * one_repeat_cnt + halfA * WIDTH);

            asc_exp_sub(exp_reg, src_reg, max_reg, mask_full);
            asc_exp_sub(exp_reg1, src_reg1, max_reg1, mask_full);
            asc_storealign(exp_ub + i * WIDTH + j * one_repeat_cnt, exp_reg, mask_full);
            asc_storealign(exp_ub + i * WIDTH + j * one_repeat_cnt + halfA * WIDTH, exp_reg1, mask_full);
        }
    }

    // 同步：前置步骤中写入exp_ub的操作完成后才能启动后续步骤。
    asc_mem_bar(VST_VLD);

    // 第二部分: ReduceSum -> Div
    for (uint16_t i = 0; i < halfA; i++) {
        asc_duplicate_scalar(sum_reg, 0.0, mask_full);
        asc_duplicate_scalar(sum_reg1, 0.0, mask_full);
        for (uint16_t j = 0; j < repeat_times; j++) {
            asc_loadalign(src_reg, exp_ub + i * WIDTH + j * one_repeat_cnt);
            asc_loadalign(src_reg1, exp_ub + i * WIDTH + j * one_repeat_cnt + halfA * WIDTH);
            asc_add(sum_reg, sum_reg, src_reg, mask_full);
            asc_add(sum_reg1, sum_reg1, src_reg1, mask_full);
        }
        asc_reduce_sum(sum_reg, sum_reg, mask_full);
        asc_reduce_sum(sum_reg1, sum_reg1, mask_full);
        asc_duplicate(sum_reg, sum_reg, mask_full);
        asc_duplicate(sum_reg1, sum_reg1, mask_full);
        for (uint16_t j = 0; j < repeat_times; j++) {
            asc_loadalign(max_reg, exp_ub + i * WIDTH + j * one_repeat_cnt);
            asc_loadalign(max_reg1, exp_ub + i * WIDTH + j * one_repeat_cnt + halfA * WIDTH);
            asc_div(div_reg, max_reg, sum_reg, mask_full);
            asc_div(div_reg1, max_reg1, sum_reg1, mask_full);
            asc_storealign(dst_ub + i * WIDTH + j * one_repeat_cnt, div_reg, mask_full);
            asc_storealign(dst_ub + i * WIDTH + j * one_repeat_cnt + halfA * WIDTH, div_reg1, mask_full);
        }
    }
}

// ======================= 核函数 V1 =========================
__global__ __vector__ void softmax_custom(__gm__ uint8_t* x, __gm__ uint8_t* y)
{
    asc_init();

    // 申请UB内存
    __ubuf__ float src0_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float dst0_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float exp0_ub[SINGLE_VF_DATA_LEN];

    __ubuf__ float src1_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float dst1_ub[SINGLE_VF_DATA_LEN];
    __ubuf__ float exp1_ub[SINGLE_VF_DATA_LEN];

    uint8_t mutex_id_src0 = 1;   // src0_ub对应的资源锁
    uint8_t mutex_id_dst0 = 2;   // dst0_ub对应的资源锁

    uint8_t mutex_id_src1 = 3;   // src1_ub对应的资源锁
    uint8_t mutex_id_dst1 = 4;   // dst1_ub对应的资源锁

    __gm__ float* x_gm = reinterpret_cast<__gm__ float*>(x) + block_idx * SINGLE_CORE_DATA_LEN;
    __gm__ float* y_gm = reinterpret_cast<__gm__ float*>(y) + block_idx * SINGLE_CORE_DATA_LEN;

    for (uint32_t i = 0; i < LOOP_COUNT; i++) {
        __ubuf__ float* src_ub;
        __ubuf__ float* dst_ub;
        __ubuf__ float* exp_ub;
        uint8_t mutex_id_src;   // src_ub对应的资源锁
        uint8_t mutex_id_dst;   // dst_ub对应的资源锁

        if(i % 2 == 0) {
            src_ub = src0_ub;
            dst_ub = dst0_ub;
            exp_ub = exp0_ub;
            mutex_id_src = mutex_id_src0;
            mutex_id_dst = mutex_id_dst0;
        } else {
            src_ub = src1_ub;
            dst_ub = dst1_ub;
            exp_ub = exp1_ub;
            mutex_id_src = mutex_id_src1;
            mutex_id_dst = mutex_id_dst1;
        }

        // 搬入操作需要独占src_ub
        asc_lock(PIPE_MTE2, mutex_id_src);
        asc_copy_gm2ub_align(src_ub, &x_gm[i * SINGLE_VF_DATA_LEN], 1, SINGLE_VF_DATA_LEN * sizeof(float),
            0, 0, false, 4, 0, 0);
        asc_unlock(PIPE_MTE2, mutex_id_src);

        // 计算操作需要独占src_ub和dst_ub
        asc_lock(PIPE_V, mutex_id_src);
        asc_lock(PIPE_V, mutex_id_dst);
        softmax_vf(dst_ub, src_ub, exp_ub);
        // asc_vf_call<softmax_vf>(dst_ub, src_ub, exp_ub);
        asc_unlock(PIPE_V, mutex_id_src);
        asc_unlock(PIPE_V, mutex_id_dst);

        // 搬出操作需要独占dst_ub
        asc_lock(PIPE_MTE3, mutex_id_dst);
        asc_copy_ub2gm_align(&y_gm[i * SINGLE_VF_DATA_LEN], dst_ub, 1, SINGLE_VF_DATA_LEN * sizeof(float),
            4, 0, 0);
        asc_unlock(PIPE_MTE3, mutex_id_dst);
    }
}

### 3.2 Host侧实现不变：

In [ ]:
%%writefile src/softmax.asc

#include <iostream>
#include <vector>
#include <iterator>
#include <random>
#include "acl/acl.h"
#include "softmax.h"

// ======================= 构造随机数据 =========================
float random_value()
{
    static std::random_device rd;
    static std::mt19937_64 gen(rd());
    static std::uniform_real_distribution<double> dist(0.0, 1.0); // 模板随机 value [0,1]
    return static_cast<float>(dist(gen));
}

void generate_test_data(std::vector<float>& values, uint32_t num, uint32_t value_dim)
{
    values.resize(num * value_dim);

    for (uint32_t i = 0; i < num; ++i) {
        for (uint32_t j = 0; j < value_dim; ++j) {
            values[i * value_dim + j] = random_value();
        }
    }
}

// ======================= 构造golden数据 =========================
std::vector<float> softmax_reference(std::vector<float>& src, uint32_t height, uint32_t width)
{
    // 支持 float 和 half 数据类型。half 类型先 cast 为 float 进行计算。
    std::vector<float> dst(src.size());
    for (uint32_t row = 0; row < height; row++) {
        uint32_t offset = row * width;
        // 找最大值
        float max_val = static_cast<float>(src[offset]);
        for (uint32_t col = 1; col < width; col++) {
            max_val = std::max(max_val, static_cast<float>(src[offset + col]));
        }
        // 计算 exp(x - max) 并求和
        float sum_val = 0.0f;
        for (uint32_t col = 0; col < width; col++) {
            float tmp = std::exp(static_cast<float>(src[offset + col]) - max_val);
            dst[offset + col] = static_cast<float>(tmp);
            sum_val += tmp;
        }
        // 归一化
        for (uint32_t col = 0; col < width; col++) {
            dst[offset + col] = static_cast<float>(static_cast<float>(dst[offset + col]) / sum_val);
        }
    }
    return dst;
}

// ======================= 校验输出结果 =========================
uint32_t verify_result(std::vector<float>& output, std::vector<float>& golden)
{
    auto print_func = [](std::vector<float>& tensor, const char* name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };

    std::cout << "Output size" << ": " << output.size() << std::endl;
    std::cout << "Golden size" << ": " << golden.size() << std::endl;
    print_func(output, "Output");
    print_func(golden, "Golden");
    constexpr float epsilon = 1e-4f;
    bool passed = true;
    uint32_t errCount = 0;
    for (size_t i = 0; i < golden.size(); i++) {
        if (std::abs(static_cast<float>(output[i]) - static_cast<float>(golden[i])) > epsilon) {
            passed = false;
            errCount++;
        }
    }
    if (passed) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed! errCount:" << errCount << std::endl;
        return 1;
    }
}

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t blocks_num = 48;
    constexpr uint32_t height = SINGLE_VF_HEIGHT * LOOP_COUNT * blocks_num;
    constexpr uint32_t width = WIDTH;
    constexpr uint32_t totalLen = height * width;

    std::vector<float> src(totalLen);
    generate_test_data(src, height, width);
    size_t inputSize = totalLen * sizeof(float);
    uint8_t* src_device = nullptr;
    uint8_t* dst_device = nullptr;
    
    aclInit(nullptr);
    aclrtSetDevice(0);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);
    
    aclrtMalloc((void**)&src_device, inputSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&dst_device, inputSize, ACL_MEM_MALLOC_HUGE_FIRST);
    
    aclrtMemcpy(src_device, inputSize, src.data(), inputSize, ACL_MEMCPY_HOST_TO_DEVICE);
    
    softmax_custom<<<blocks_num, 0, stream>>>(src_device, dst_device);
    aclrtSynchronizeStream(stream);
    
    std::vector<float> output(totalLen);
    aclrtMemcpy(output.data(), inputSize, dst_device, inputSize, ACL_MEMCPY_DEVICE_TO_HOST);
    
    std::vector<float> golden = softmax_reference(src, height, width);
    uint32_t result = verify_result(output, golden);
    
    aclrtFree(src_device);
    aclrtFree(dst_device);
    
    aclrtDestroyStream(stream);
    aclrtResetDevice(0);
    aclFinalize();
    return result;
}

### 3.3 CMake编译配置文件不变：

In [ ]:
%%writefile src/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode: npu, sim")
set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU architecture: dav-3510")

find_package(ASC REQUIRED)
project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo softmax.asc)

target_link_libraries(demo PRIVATE m)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)

### 3.4 编译和运行

In [ ]:
!cd src && mkdir -p build
!cd src/build/ && \
cmake .. && \
make && \
./demo

## 4. 优化前后的性能对比

**图3** 高阶版Softmax算子流水图

<img src="./images/07.03_softmax/softmax_advanced_timeline.png" title="高阶版本流水图" style="zoom:100%;" />

详细性能数据：
|   | vector核总时间 | PIPE_V时间 | PIPE_V时间占比 | PIPE_S时间 | PIPE_S时间占比 | PIPE_MTE2时间 | PIPE_MTE2时间占比 | PIPE_MTE3时间 | PIPE_MTE3时间占比 |
| :-----: | :-------: | :-------: | :-------: | :-------: | :-------: | :-------: | :-------: | :-------: | :-------: |
| 进阶版本 | 4261.62  | 643.671 | 15.1% | 552.156 | 13% | 2938.025 | 68.9% | 1080.408 | 25.4% |
| 高阶版本 | 2653.4  | 721.774 | 27.2% | 73.479 | 2.8% | 2562.36 | 96.6% | 1544.465 | 58.2% |

数据总结：
- 整体性能提升37.74%。
- PIPE_MTE2时间占比明显提升，达到96.58%。各流水的并行度有极大提升。收益主要来自于Double Buffer。
- PIPE_MTE2/PIPE_MTE3/PIPE_S耗时均有大幅下降。收益主要来自于一次性读写整个VF所需数据，以及关闭l2 cache。
- PIPE_V耗时略有提升，主要原因是单VF处理数据量下降，降低了VF内部双发带来的收益。
- PIPE_V耗时略有提升，但每个周期内PIPE_V耗时仍明显小于PIPE_MTE2。各流水更加均衡，整体性能收益明显。

---
## 5. 课后实践

现有一组4194304 * 512的float类型的数据，请独立编写一个softmax算子。可探索不同的数据切分方式, 并注意单核UB内存空间上限。

用户需完成如下3个文件：

1、编写softmax.asc：在该文件中完成Device侧核函数和VF函数的实现。

In [ ]:
%%writefile src/softmax.asc

2、编写softmax.h：在该文件中完成Host侧的功能。如测试数据构造、结果校验、核函数调用等。

In [ ]:
%%writefile src/softmax.h

3、编写CMakeLists.txt：在该文件中完成CMake配置。

In [ ]:
%%writefile src/CMakeLists.txt

执行如下脚本可验证精度是否符合预期：

In [ ]:
!cd src && mkdir -p build && cd build && \
cmake .. && \
make && \
./demo

执行如下命令可查看参考答案：

In [ ]:
!cat answer/07.03_softmax/softmax.asc

In [ ]:
!cat answer/07.03_softmax/softmax.h

In [ ]:
!cat answer/07.03_softmax/CMakeLists.txt